In [80]:
##### Compiles csv of country averages to use in relative predictor calculations

import os
import pandas as pd
from pathlib import Path
import numpy as np
from functools import reduce

In [81]:
### Set-up

# Get the current working directory
cd = Path.cwd().parent 

# folder of predictors 
predictors = Path(f'{cd}/Data/Clean/Predictors/Vectors/Country_averages')

# intensities 
intensities = pd.read_csv(f"{cd}/Data/Clean/Intensities/country_intensities.csv")

# correspondance table
countries = pd.read_csv(f'{cd}/Data/Correspondence_tables/country_names.csv', encoding="cp1252")

# save path
save_path = f'{cd}/Data/Clean/Predictors/country_averages.csv'

In [82]:
##### Merge all predictors into one df (dropping all duplicate GID_0 entries first)
dfs = []
for f in os.listdir(predictors):
    if f.endswith(".csv"):
        df = pd.read_csv(os.path.join(predictors, f))
        dupes = df['GID_0'].duplicated().sum()
        if dupes > 0:
            df = df.drop_duplicates(subset='GID_0')
        dfs.append(df)

predictors_df = reduce(lambda left, right: pd.merge(left, right, on='GID_0', how='outer'), dfs)

In [83]:
##### Merge with list of countries needed 

# merge with ISO codes
predictors_df = predictors_df.merge(countries, left_on='GID_0', right_on='SHP_code', how='left')

# merge with list of countries with intensities 
total_df = intensities.merge(predictors_df, on='ISO3', how='left')

In [84]:
#### Fill missing data with subregion averages 

# columns to average
average_col = ['SOC', 'pct_cropland_irrigated', 'female_share',
       'pop_density_people_per_km2', 'average_travel_time_city',
       'average_travel_time_port', 'probability_economic_land_use_objective',
       'probability_survival_land_use_objective', 'growing_season_length_days',
       'child_dependency_ratio', 'GDP_pc', 'USD_production_per_HA',
       'tonnes_production_per_HA', 'slope', 'buffalo_density_per_km2',
       'chicken_density_per_km2', 'cattle_density_per_km2',
       'goats_density_per_km2', 'pigs_density_per_km2',
       'sheep_density_per_km2', 'livestock_density_LU_per_km2',
       'cereals_share_production_USD', 'fibres_share_production_USD',
       'fruits_share_production_USD', 'oilcrops_share_production_USD',
       'pulses_share_production_USD', 'roots_tubers_share_production_USD',
       'rest_of_crops_share_production_USD',
       'sugar_crops_share_production_USD', 'vegetables_share_production_USD',
       'rubber_share_production_USD', 'ruminants_share_production_USD',
       'monogastrics_share_production_USD', 'poultry_share_production_USD',
       'other_share_production_USD', 'pct_GDP_ag', 'share_vlarge_field',
       'share_large_field', 'share_medium_field', 'share_small_field',
       'share_vsmall_field', 'share_with_nightlights', 'crop_intensity'
       ]

subregion_avg_full = total_df[average_col + ['UN_subregion']]
region_avg_full = total_df[average_col + ['UN_region']]

# get average by subregion 
subregion_avg = subregion_avg_full.groupby('UN_subregion').mean()
subregion_avg = subregion_avg.add_prefix('subregion_')

region_avg = region_avg_full.groupby('UN_region').mean()
region_avg = region_avg.add_prefix('region_')

# join back to main df 
total_df = total_df.merge(subregion_avg, on='UN_subregion', how='left')
total_df = total_df.merge(region_avg, on='UN_region', how='left')

# fill na's with subregion and then region 
for col in average_col:
    total_df[col] = total_df[col].fillna(total_df[f'subregion_{col}'])
    total_df[col] = total_df[col].fillna(total_df[f'region_{col}'])

# filter back to core columns
total_df = total_df[['ISO3'] + average_col]

In [85]:
### Save
total_df.to_csv(save_path, index=False)